In [0]:
# =============================================================
# silver_reader.py
# Layer     : Silver
# Purpose   : Reads from Bronze Delta only.
#             Handles incremental filtering using the
#             _load_date lineage column injected by Bronze.
#
# STRICT RULE: No writes. No cleaning logic. No schemas.
#              If you see withColumn here → wrong file.
# =============================================================

from pyspark.sql.functions import col


def _normalize_bronze_load_date(load_date: str) -> str:
    """
    Bronze stores `_load_date` as a partition-style string like
    `year=2025/month=01/day=14`, while orchestrators pass `yyyy-mm-dd` or timestamps.
    """
    if not load_date:
        return load_date
        
    if load_date.startswith("year="):
        return load_date

    # Fix: Extract only the first 10 characters (yyyy-MM-dd) to strip out 'T00:00:00Z'
    clean_date = load_date[:10]

    parts = clean_date.split("-")
    if len(parts) == 3 and all(parts):
        year, month, day = parts
        return f"year={year}/month={month}/day={day}"

    return load_date


def read_bronze(table_name: str, table_type: str,
                load_date: str, is_incremental: bool):
    """
    Read Bronze Delta for the given table.

    For incremental fact loads:
        Filters to rows where _load_date == load_date.
        This ensures Silver only processes today's new Bronze rows
        without needing a separate watermark table.

    For dimensions or full refresh:
        Reads all rows — dimensions are always fully refreshed.

    Args:
        table_name     : e.g. "device_snapshots"
        table_type     : "dimension" or "fact"
        load_date      : "yyyy-mm-dd" from ADF widget
        is_incremental : True for facts, False for dimensions

    Returns:
        DataFrame from Bronze Delta (unmodified)
    """
    bronze_path = abfss("bronze", table_name)
    print(f"[silver_reader] Reading Bronze: {bronze_path}")

    df = spark.read.format("delta").load(bronze_path)
    print(f"[silver_reader] Bronze rows before filtering: {df.count():,}")

    if table_type == "fact" and is_incremental:
        normalized_load_date = _normalize_bronze_load_date(load_date)
        print(f"[silver_reader] Requested load_date widget value: {load_date}")
        print(f"[silver_reader] Normalized Bronze _load_date: {normalized_load_date}")

        filtered_df = df.filter(col("_load_date") == normalized_load_date)
        matched_count = filtered_df.count()
        print(f"[silver_reader] Rows matched for normalized _load_date: {matched_count:,}")

        if matched_count == 0:
            print("[silver_reader] No rows matched. Latest available Bronze _load_date values:")
            (df.groupBy("_load_date")
               .count()
               .orderBy(col("_load_date").desc())
               .show(5, truncate=False))

        df = filtered_df

    row_count = df.count()
    print(f"[silver_reader] Rows read from Bronze after filtering: {row_count:,}")

    return df


def drop_lineage_columns(df):
    """
    Remove Bronze audit columns before writing to Silver.
    Silver adds its own _silver_load_timestamp separately.

    These columns are useful for debugging Bronze but should
    not pollute Silver's clean analytical schema.
    """
    lineage_cols = [
        "_bronze_load_timestamp",
        "_source_file",
        "_load_date",
        "_table_name",
        "_table_type",
        "_is_quarantined",
    ]
    existing = [c for c in lineage_cols if c in df.columns]
    print(f"[silver_reader] Dropping lineage columns: {existing}")
    return df.drop(*existing)

print("[silver_reader] Loaded.")